In [2]:
from google.cloud import bigquery
import pandas as pd

from pathlib import Path
import sys
sys.path.append(str(Path('../python').resolve()))

from run_youtube_pipeline import extract_pipeline

# Define Client

In [3]:
client = bigquery.Client()  # uses gcloud credentials automatically

## Clean Data

In [94]:
user_name = "TaylorSwift"
max_videos=5
info_mv = extract_pipeline(user_name, max_videos)


In [102]:
info_mv

[{'etag': 'fpHY3xk0WzFvihSqUggnRLthxvc',
  'id': 'w_DD-7_zVtw',
  'title': 'And, baby, that’s show business for you. New album The Life of a Showgirl. Out October 3\xa0 ❤️\u200d🔥',
  'published': '2025-08-14T05:00:58Z',
  'description': '',
  'link': 'https://www.youtube.com/watch?v=w_DD-7_zVtw',
  'thumbnails': 'https://i.ytimg.com/vi/w_DD-7_zVtw/hqdefault.jpg'},
 {'etag': 'laHqQ7cAS7ySrB0qZm2Tnnh6ZNE',
  'id': 'P0haCYjysUs',
  'title': 'Anti-Hero (Behind The Scenes with The Ghosts In The Room)',
  'published': '2024-12-13T17:24:03Z',
  'description': 'Behind the scenes of "Anti-Hero" official music video with The Ghosts In The Room, from the album \'Midnights\' Watch the ...',
  'link': 'https://www.youtube.com/watch?v=P0haCYjysUs',
  'thumbnails': 'https://i.ytimg.com/vi/P0haCYjysUs/hqdefault.jpg'},
 {'etag': 'feRHhpOZd0x9MRM4mwHCaEbNvfA',
  'id': 'PQp643val70',
  'title': 'cardigan (behind the scenes - forest &amp; ocean)',
  'published': '2024-12-13T15:26:29Z',
  'description': 'B

In [103]:
df = pd.DataFrame(info_mv)

In [113]:
df.shape

(5, 7)

In [114]:
if len(info_mv) != df.shape[0]:
    Exception("Not matching entries from list -> df")

In [115]:
# df.info()
#pd.isna(df.description.iloc[0])
#df_mv[df_mv.isna().any(axis=1)]

In [116]:
# Replace NaN, empty strings, or "NULL" (case-insensitive) with "None"
df['description'] = df['description'].apply(
    lambda x: "None" if (pd.isna(x) or str(x).strip() == "" or str(x).strip().upper() == "NULL") else x
)


In [117]:
#missing values aren’t converted to "nan" but instead to "None"
df = df.fillna("None").astype(str)
df = df.astype(str)
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)


In [118]:
import numpy as np
# Replace string "None" with real NaN
df['published'] = df['published'].replace("None", np.nan)

# Convert to datetime, invalid values will become NaT
df['published'] = pd.to_datetime(df['published'], errors='coerce')

In [119]:
df

,etag,id,title,published,description,link,thumbnails
0,fpHY3xk0WzFvihSqUggnRLthxvc,w_DD-7_zVtw,"And, baby, that’s show business for you. New a...",2025-08-14 05:00:58+00:00,None,https://www.youtube.com/watch?v=w_DD-7_zVtw,https://i.ytimg.com/vi/w_DD-7_zVtw/hqdefault.jpg
1,laHqQ7cAS7ySrB0qZm2Tnnh6ZNE,P0haCYjysUs,Anti-Hero (Behind The Scenes with The Ghosts I...,2024-12-13 17:24:03+00:00,"Behind the scenes of ""Anti-Hero"" official musi...",https://www.youtube.com/watch?v=P0haCYjysUs,https://i.ytimg.com/vi/P0haCYjysUs/hqdefault.jpg
2,feRHhpOZd0x9MRM4mwHCaEbNvfA,PQp643val70,cardigan (behind the scenes - forest &amp; ocean),2024-12-13 15:26:29+00:00,"Behind the scenes of ""cardigan"" official music...",https://www.youtube.com/watch?v=PQp643val70,https://i.ytimg.com/vi/PQp643val70/hqdefault.jpg
3,uGOjeZyN-N4bLk0vbng_Qsn6Tjk,K-8dOw7yuPo,Bejeweled (Behind the Scenes with Laura Dern),2024-12-13 15:03:35+00:00,"Behind the scenes of ""Bejeweled"" official musi...",https://www.youtube.com/watch?v=K-8dOw7yuPo,https://i.ytimg.com/vi/K-8dOw7yuPo/hqdefault.jpg
4,QLa4fe4tVxefOvV6lyapCiGregw,-ddfFsLHNQs,Anti-Hero (Behind the Scenes with Mike Birbigl...,2024-12-13 15:04:17+00:00,"Behind the scenes of ""Anti-Hero"" official musi...",https://www.youtube.com/watch?v=-ddfFsLHNQs,https://i.ytimg.com/vi/-ddfFsLHNQs/hqdefault.jpg


In [ ]:
df['published'] = pd.to_datetime(df[~df['published'].isna()][['published']])

0   2025-08-14 05:00:58+00:00
1   2024-12-13 17:24:03+00:00
Name: published, dtype: datetime64[ns, UTC]

In [ ]:
for i in range(len(info_mv)):
    info = info_mv[i]
    

# Read Tables

In [5]:
def read_bq_table(project_id, dataset_id, table_id, max_records=10):
    
    query = f"SELECT * FROM `{project_id}.{dataset_id}.{table_id}` LIMIT {max_records}"
    df = client.query(query).to_dataframe()
    return df


In [ ]:
project_id = "swiftie-friend" 
dataset_id = "album_songs_summary"
table_id = "album_songs_summary"
max_records = 10
df = read_bq_table(project_id, dataset_id, table_id, max_records)
df.head(1)

/home/mrosaria/Projects/NLP/SwiftieFriend/swiftie_env/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,track_name,track_id,duration_ms,explicit,track_number,album_id,album_name,album_release_date,album_total_tracks,album_type,artist_name,artist_id,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence
0,imgonnagetyouback,1kcwpPDQnqEqmezzXdJTCP,222072,False,18,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.784,0.391,-9.471,0.0633,0.6080,0.000000,0.2520,0.150
1,The Albatross,4EF6IyONolQy0bIQXm2EmX,183878,False,19,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.352,0.479,-8.942,0.0583,0.6550,0.000000,0.0935,0.292
2,Chloe or Sam or Sophia or Marcus,1rmEsOezwf2lmIZTMAO5Ag,213281,False,20,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.516,0.451,-8.552,0.0446,0.7800,0.000000,0.1050,0.218
3,How Did It End?,5Bedn0svl0ZD7RGmJkmKKw,238706,False,21,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.492,0.379,-8.859,0.0257,0.7850,0.000930,0.1080,0.273
4,So High School,7Mts0OfPorF4iwOomvfqn1,228800,False,22,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.366,0.866,-4.514,0.0466,0.0274,0.000003,0.1060,0.293
5,I Hate It Here,3hlGuz3loYoLfI3bpwieWq,243875,False,23,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.485,0.562,-7.197,0.0346,0.7170,0.000000,0.3570,0.297
6,I Look in People's Windows,1Zai5UJ2di3qEuR2HeT2s8,131907,False,25,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.763,0.274,-11.429,0.0738,0.6900,0.000000,0.1640,0.385
7,The Prophecy,18WFFUIsewmA8g31KAeo3e,249807,False,26,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.421,0.525,-10.359,0.0472,0.8250,0.000003,0.3080,0.535
8,Peter,3zMDGj4D8ogaYgAIZPeU7S,283957,False,28,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.342,0.376,-8.212,0.0372,0.8050,0.000000,0.0593,0.290
9,Robin,2CnjDMdpRjlWv04Xk3s6MW,240893,False,30,5H7ixXZfsNMGbIE5OBSpcb,THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY,2024-04-19,31,album,Taylor Swift,06HL4z0CvFAxyc27GXpf02,0.476,0.386,-10.601,0.0300,0.7910,0.000423,0.1000,0.158


In [4]:
client = bigquery.Client()  # uses gcloud credentials automatically

# Create New Table
Create first the dataset on UI if doesnt exist

## Load data by UPLOADING local file

In [9]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,  # skip header row
    autodetect=True,      # let BigQuery detect schema
)

project_id = "swiftie-friend" 
dataset_id = "social_media"
table_id = "youtube_music_videos"
table_ref = f"{project_id}.{dataset_id}.{table_id}"

input_path="../data/"
input_file_name="youtube_taylor_last_2_mv"
#df = pd.read_csv(f"{input_path}{input_file_name}.csv")

with open(f"{input_path}{input_file_name}.csv", "rb") as source_file:
    job = client.load_table_from_file(source_file, table_ref, job_config=job_config)
    job.result()  # Wait for completion



In [10]:
read_bq_table(project_id, dataset_id, table_id, max_records)

/home/mrosaria/Projects/NLP/SwiftieFriend/swiftie_env/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,etag,id,title,published,description,link,thumbnails
0,fpHY3xk0WzFvihSqUggnRLthxvc,w_DD-7_zVtw,"And, baby, that’s show business for you. New a...",2025-08-14 05:00:58+00:00,None,https://www.youtube.com/watch?v=w_DD-7_zVtw,https://i.ytimg.com/vi/w_DD-7_zVtw/hqdefault.jpg
1,laHqQ7cAS7ySrB0qZm2Tnnh6ZNE,P0haCYjysUs,Anti-Hero (Behind The Scenes with The Ghosts I...,2024-12-13 17:24:03+00:00,"Behind the scenes of ""Anti-Hero"" official musi...",https://www.youtube.com/watch?v=P0haCYjysUs,https://i.ytimg.com/vi/P0haCYjysUs/hqdefault.jpg


## Load data, Define Table and its Schema

In [8]:
# project_id = "swiftie-friend" 
# dataset_id = "social_media"
# table_id = "youtube_music_videos"
# max_records = 10
# df = read_bq_table(project_id, dataset_id, table_id, max_records)
# df.head(1)

In [30]:
input_path="../data/"
input_file_name="youtube_taylor_last_2_mv"
df = pd.read_csv(f"{input_path}{input_file_name}.csv")


In [33]:
df[df.description.isna()]

,etag,id,title,published,description,link,thumbnails
0,fpHY3xk0WzFvihSqUggnRLthxvc,w_DD-7_zVtw,"And, baby, that’s show business for you. New a...",2025-08-14T05:00:58Z,NaN,https://www.youtube.com/watch?v=w_DD-7_zVtw,https://i.ytimg.com/vi/w_DD-7_zVtw/hqdefault.jpg


In [49]:
str(df.description.iloc[0])

'nan'

nan

In [36]:
df = df.description.apply(lambda x: "No description provided" if x.isnan() else str(x))

AttributeError: 'float' object has no attribute 'isnan'

In [ ]:
df = df.apply(lambda col: type(col.dropna().iloc[0]))


,etag,id,title,published,description,link,thumbnails
0,fpHY3xk0WzFvihSqUggnRLthxvc,w_DD-7_zVtw,"And, baby, that’s show business for you. New a...",2025-08-14T05:00:58Z,NaN,https://www.youtube.com/watch?v=w_DD-7_zVtw,https://i.ytimg.com/vi/w_DD-7_zVtw/hqdefault.jpg
1,laHqQ7cAS7ySrB0qZm2Tnnh6ZNE,P0haCYjysUs,Anti-Hero (Behind The Scenes with The Ghosts I...,2024-12-13T17:24:03Z,"Behind the scenes of ""Anti-Hero"" official musi...",https://www.youtube.com/watch?v=P0haCYjysUs,https://i.ytimg.com/vi/P0haCYjysUs/hqdefault.jpg


In [32]:
for col in df.columns:
    print(col, type(df[col].iloc[0]) )


etag <class 'str'>
id <class 'str'>
title <class 'str'>
published <class 'str'>
description <class 'float'>
link <class 'str'>
thumbnails <class 'str'>


In [21]:
#df.info()
df.dtypes

etag           object
id             object
title          object
published      object
description    object
link           object
thumbnails     object
dtype: object

In [ ]:
df.apply(lambda col: type(col.dropna().iloc[0]))


In [ ]:
project_id = "swiftie-friend" 
dataset_id = "social_media"
table_id = "youtube_music_videos_v2"
table_ref = f"{project_id}.{dataset_id}.{table_id}"

schema = [
    bigquery.SchemaField("etag", "STRING"),
    bigquery.SchemaField("id", "STRING"),
    bigquery.SchemaField("title", "STRING"),
    bigquery.SchemaField("published", "TIMESTAMP"),
    bigquery.SchemaField("description", "STRING"),
    bigquery.SchemaField("link", "STRING"),
    bigquery.SchemaField("thumbnails", "STRING"),
]

table = bigquery.Table(table_ref, schema=schema)
table = client.create_table(table, exists_ok=True)  # overwrite if exists
print(f"Table {table.table_id} created with schema.")